In [1]:
import hopsworks
import pandas as pd
from scraper import FirehoseScraper
from feature_transform import FeatureConfig, build_feature_table, select_feature_store_columns, calculate_trend_features
from scrape_data_transform import build_csv_data
import os
from datetime import datetime
import re

/Users/jukeliu/python_workspace/trends-prediction-project/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
time_var = datetime.now().strftime('%Y%m%d_%H%M%S')

jsonl_folder_path = "data"
os.makedirs(jsonl_folder_path, exist_ok=True)
output_file_name = f"{jsonl_folder_path}/bluesky_posts_{time_var}.jsonl"
print("starting data scrape...")
# scrape data for 5 minutes
archiver = FirehoseScraper(output_file=output_file_name, verbose=False, num_workers=4)
archiver.start_collection(duration_seconds=5, post_limit=None)

print("transforming scraped data into feature csv...")
build_csv_data(input_path = jsonl_folder_path, output_csv = "feature_data.csv")

df = pd.read_csv('train_data/feature_data.csv',
                    dtype={
        "Trend": "string",
        "source_file": "string"
    },
                parse_dates=["time_stamp"]
)

print("building feature table...")
english_pattern = re.compile(r'^#[A-Za-z0-9_]+$')
df_en = df[df["Trend"].str.match(english_pattern)]
df_en.head()

cfg = FeatureConfig(bucket_minutes=5, rolling_windows=(3, 12))
features_df = build_feature_table(df_en, cfg)
features_df = select_feature_store_columns(features_df)
print(features_df.head())
print(features_df.info())

project = hopsworks.login(api_key_value="CNgidWirCRs6p66s.GjTJhC5kmU5qZnGvt4QR5VjDiwU5XgKZeGtjvPojyxFhkAzxgOlEtDxCaYFnh0Ge")
fs = project.get_feature_store()
trends_fg = fs.get_feature_group("trends_feature_store", version=2)

trends_df = trends_fg.read()
print(trends_df.info())

starting data scrape...
Starting collection...


/Users/jukeliu/python_workspace/trends-prediction-project/.venv/lib/python3.11/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'default' attribute with value None was provided to the `Field()` function, which has no effect in the context it was used. 'default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/Users/jukeliu/python_workspace/trends-prediction-project/.venv/lib/python3.11/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'default' attribute with value None was provided to the `Field()` function, which has no effect in the context it was used. 'default' is field-specific metadata, and can only be attached to a model field using 


Time limit reached.

Collection complete!
Collected 78 posts in 5.08 seconds
Average rate: 15.4 posts/sec
Output saved to: data/bluesky_posts_20260111_174942.jsonl

Collection complete!
Collected 78 posts in 5.08 seconds
Average rate: 15.4 posts/sec
Output saved to: data/bluesky_posts_20260111_174942.jsonl
transforming scraped data into feature csv...
building feature table...
                trend                  ts  post_count  count_lag1  \
0                #gay 2026-01-11 14:40:00           2         NaN   
1               #lust 2026-01-11 14:40:00           2         NaN   
2            #passion 2026-01-11 14:40:00           2         NaN   
3            #bluesky 2026-01-11 14:40:00           1         NaN   
4  #buildingmaterials 2026-01-11 14:40:00           1         NaN   

   delta_count  share_of_attention  rank_in_snapshot  tokens_per_post  \
0          NaN            0.060606                 1             21.0   
1          NaN            0.060606                 2      

In [3]:
trends_df["trend"] = trends_df["trend"].astype("string")
trends_df.head()

,trend,ts,post_count,count_lag1,delta_count,share_of_attention,rank_in_snapshot,tokens_per_post,hour_sin,hour_cos,label_count_next,label_delta_next,trend_raw,source_file,token_volume
0,#rant,2026-01-06 13:00:00+00:00,9,NaN,NaN,0.018828,16,57.000000,-0.258819,-0.965926,NaN,NaN,#rant,bluesky_20260106_130155.jsonl,513
1,#allgays,2026-01-10 18:05:00+00:00,9,10.0,-1.0,0.022500,21,38.000000,-0.999762,0.021815,8.0,-1.0,#allgays,bluesky_20260110_180540.jsonl,342
2,#envtuber,2026-01-05 21:05:00+00:00,7,7.0,0.0,0.018970,24,51.571429,-0.691513,0.722364,7.0,0.0,#envtuber,bluesky_20260105_210849.jsonl,361
3,#maduro,2026-01-06 01:20:00+00:00,8,7.0,1.0,0.030303,15,17.750000,0.342020,0.939693,12.0,4.0,#maduro,bluesky_20260106_012429.jsonl,142
4,#gay,2026-01-10 10:00:00+00:00,13,22.0,-9.0,0.025243,15,50.769231,0.500000,-0.866025,14.0,1.0,#gay,bluesky_20260110_100233.jsonl,660


In [4]:
features_df["ts"] = (
    features_df["ts"]
        .dt.tz_localize("UTC")      # make it timezone-aware
        .dt.tz_convert("Etc/UTC")   # normalize to Etc/UTC (same offset, different name)
        .dt.as_unit("us")          # convert ns → µs
)

In [5]:
features_df.head()
features_df.info()
trends_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 186 entries, 0 to 185
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype                  
---  ------              --------------  -----                  
 0   trend               186 non-null    string                 
 1   ts                  186 non-null    datetime64[us, Etc/UTC]
 2   post_count          186 non-null    int64                  
 3   count_lag1          2 non-null      float64                
 4   delta_count         2 non-null      float64                
 5   share_of_attention  186 non-null    float64                
 6   rank_in_snapshot    186 non-null    int32                  
 7   tokens_per_post     186 non-null    float64                
 8   hour_sin            186 non-null    float64                
 9   hour_cos            186 non-null    float64                
 10  label_count_next    2 non-null      float64                
 11  label_delta_next    2 non-null      float64  

In [6]:
combined_df = pd.concat([features_df, trends_df]).reset_index(drop=True)
print(combined_df.info())

final_df = calculate_trend_features(combined_df)
# print("final feature table:")
print(final_df.info())
print(final_df.head())
# final_df.to_csv('train_data/final_feature_data.csv', index=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29048 entries, 0 to 29047
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype                  
---  ------              --------------  -----                  
 0   trend               29048 non-null  string                 
 1   ts                  29048 non-null  datetime64[us, Etc/UTC]
 2   post_count          29048 non-null  int64                  
 3   count_lag1          24495 non-null  float64                
 4   delta_count         24495 non-null  float64                
 5   share_of_attention  29048 non-null  float64                
 6   rank_in_snapshot    29048 non-null  int32                  
 7   tokens_per_post     29048 non-null  float64                
 8   hour_sin            29048 non-null  float64                
 9   hour_cos            29048 non-null  float64                
 10  label_count_next    24495 non-null  float64                
 11  label_delta_next    24495 non-null  float